# Session 2: RAG & Retrieval

**AI Developer Training**

In Session 1 you extracted structured data out of a submission the model already had in front of it. Today you'll build a pipeline that gives the model documents it was never trained on — a local set of underwriting guidelines — and have it answer questions *grounded* in that content. This is Retrieval-Augmented Generation (RAG).

Throughout the notebook you'll see:
- 📝 **Exercise** / **Try It Yourself** — write or modify something yourself
- 💬 **Reflection** — a question to think about or discuss
- 💡 **Key Takeaway** — the main point to walk away with

Same rule as Session 1: all LLM calls go through `get_client().generate(...)` / `get_client().generate_structured(...)`, and all embedding calls go through `get_client().embed(...)`. Never a raw `boto3`/`openai` SDK call.

Before starting, make sure you've completed the steps in this folder's `setup.md`.

## Section 1 — Setup & RAG Overview

**Goal:** Understand what RAG is for and see the shape of the full pipeline before building any of it.

An LLM only knows two things: what it learned during training, and whatever you put in the prompt. RAG closes that gap by retrieving relevant text from your own documents and adding it to the prompt before the model answers.

RAG has two pipelines:
- **Offline (indexing):** documents → chunk → embed → store in a vector index
- **Online (query time):** question → embed → retrieve top-K similar chunks → augment the prompt → generate an answer

Today you'll build both, end to end, against a small set of synthetic underwriting documents. These are real PDFs in this folder's `docs/` directory — formatted like actual underwriting manual pages, synthetic content, checked into the repo, no external download or account needed.

### Guided Demo

In [ ]:
from shared.llm import get_client

client = get_client()

In [ ]:
from pathlib import Path

from pypdf import PdfReader

DOCS_DIR = Path("docs")

DOC_IDS = ["GL-GUIDELINES", "PROPERTY-APPETITE", "CYBER-EXCLUSIONS", "WORKCOMP-CLASSIFICATION"]

def load_pdf_doc(path: Path) -> dict:
    """Extract title + body text from one of our underwriting-manual PDFs.

    Each PDF has a fixed layout: a page header, the document title, a
    doc-number/effective-date line, then the body text.
    """
    reader = PdfReader(path)
    raw_text = "\n".join(page.extract_text() for page in reader.pages)
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]

    title = lines[1]
    body = " ".join(lines[3:])
    return {"title": title, "text": body}

UNDERWRITING_DOCS = {}
for doc_id in DOC_IDS:
    filename = doc_id.lower() + ".pdf"
    UNDERWRITING_DOCS[doc_id] = load_pdf_doc(DOCS_DIR / filename)

for doc_id, doc in UNDERWRITING_DOCS.items():
    print(f"=== {doc_id}: {doc['title']} ===")
    print(doc["text"][:150] + "...\n")

&nbsp;

💡 **Key Takeaway** — RAG has two distinct pipelines: offline indexing (build the searchable store once) and online query (use it on every question). Today's lab builds both.

---

## Section 2 — Chunking Documents

**Goal:** Understand why documents must be split into chunks before embedding, and that chunk boundaries matter.

We can't embed and retrieve an entire document as one unit — it's too coarse (a whole document either matches or it doesn't) and often too large for the embedding model's input limit. Instead we split each document into overlapping chunks. The overlap prevents a sentence from being cut in half with no chunk containing the whole idea.

### Guided Demo

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Split text into overlapping fixed-size character windows."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += chunk_size - overlap
    return chunks

sample_text = UNDERWRITING_DOCS["PROPERTY-APPETITE"]["text"]

for size in [150, 300, 600]:
    chunks = chunk_text(sample_text, chunk_size=size, overlap=30)
    print(f"chunk_size={size} -> {len(chunks)} chunks")
    print("  first chunk:", repr(chunks[0]))
    print()

&nbsp;

### 🔧 Try It Yourself — Exercise 2.1

Run `chunk_text` against `UNDERWRITING_DOCS["GL-GUIDELINES"]["text"]` with a small chunk size (e.g. 80) and no overlap (`overlap=0`). Look at the chunk boundaries — find one that cuts off a sentence or number mid-way (e.g. splits "$1,000,000" or "senior underwriter sign-off" across two chunks).

💬 **Reflection** — What would happen at retrieval time if the only chunk that mentions "$5,000,000" doesn't also contain the phrase "General liability" or "GL"? Would a semantic search for "general liability limits" still find it?

In [ ]:
# Try a small chunk size with no overlap and inspect the boundaries
gl_text = UNDERWRITING_DOCS["GL-GUIDELINES"]["text"]

small_chunks = chunk_text(gl_text, chunk_size=80, overlap=0)
for i, c in enumerate(small_chunks):
    print(f"[{i}] {c!r}")

&nbsp;

💡 **Key Takeaway** — Chunk size is a trade-off between retrieval precision, context completeness, and token cost. There's no universally "correct" size — it depends on how self-contained each idea in your documents is.

📌 **Other chunking techniques you'll see in real RAG systems** — today's `chunk_text` splits on a fixed character count, which is simple but can cut mid-word or mid-number. Other common approaches:
- **Word-based chunking** — window measured in whole words instead of characters, so a cut never lands inside a word.
- **Sentence-based chunking** — group whole sentences together up to a size limit, so a cut never lands mid-idea, at the cost of less predictable chunk sizes.
- **Recursive/paragraph-based chunking** — split on paragraph or section boundaries first, and only fall back to a smaller splitter for oversized paragraphs (this is what LangChain's `RecursiveCharacterTextSplitter` does).
- **Semantic chunking** — embed sentences and split where meaning shifts significantly, rather than at a fixed size at all. Let's actually build this one below.

Each is a different answer to the same trade-off from the takeaway above: simplicity/speed vs. respecting the document's natural structure.

### Guided Demo — Semantic Chunking

Instead of splitting arbitrarily on character count, what if we could split on concept or thought?

Semantic chunking aims to tackle this. Here's how it works:
1. **Split the document into sentences** with a simple regex (`.`/`!`/`?` followed by whitespace). This is the mechanical step — it makes no decision about chunk boundaries, it just produces sentence-sized pieces to work with. Known limitation: a period-based regex can misfire on decimals or dollar amounts (e.g. "$1,000,000.50"), which show up throughout these documents — the same failure mode as the Exercise 2.1 chunk boundary. That's an acceptable simplification for a demo over four short, clean synthetic documents; a production system would use a real sentence tokenizer here.
2. **Embed all the sentences in one batched call** to `client.embed(sentences)` — no need for a separate call per sentence.
3. **Walk the sentences in order**, comparing each sentence's embedding to the next one's with cosine similarity. Split where similarity drops relative to *this document's own* distribution of adjacent-sentence similarities — specifically, below the Nth percentile of those similarities (default: 40th) — rather than below a fixed absolute number.

⚠️ **Why a percentile instead of a fixed threshold like 0.75?** Absolute cosine similarity values depend heavily on the embedding model — some models cluster almost everything above 0.7, others put unrelated sentences much lower and related ones only slightly higher. A hardcoded threshold tuned for one embedding model can easily never fire on another's output — every similarity falls below it, so the method degenerates into one chunk per sentence, indistinguishable from the mechanical splitting in step 1. Computing the threshold as a percentile of *each document's own* similarity spread makes the method portable across embedding models and documents, instead of tuned to one model's numeric range.

This is the same core approach used by real implementations like LlamaIndex's `SemanticSplitterNodeParser` and LangChain's `SemanticChunker` — both use a relative/percentile-based breakpoint rather than an absolute cosine value for exactly this reason.

In [ ]:
import math
import re

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

def semantic_chunk(text: str, percentile: float = 40) -> list[str]:
    """Group sentences into chunks based on embedding similarity, splitting where
    similarity drops below the given percentile of this document's own
    sentence-to-sentence similarity distribution — not a fixed size or a fixed
    absolute similarity value.
    """
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    if len(sentences) <= 1:
        return sentences

    embeddings = client.embed(sentences)
    similarities = [
        cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(sentences))
    ]

    threshold = sorted(similarities)[int(len(similarities) * percentile / 100)]
    print(f"(threshold={threshold:.3f}, the {percentile}th percentile of this document's own adjacent-sentence similarities)")

    chunks = []
    current_chunk = [sentences[0]]
    for i, similarity in enumerate(similarities, start=1):
        if similarity >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
    chunks.append(" ".join(current_chunk))
    return chunks

In [ ]:
gl_text = UNDERWRITING_DOCS["GL-GUIDELINES"]["text"]

semantic_chunks = semantic_chunk(gl_text)
print(f"{len(semantic_chunks)} semantic chunks\n")
for i, c in enumerate(semantic_chunks):
    print(f"[{i}] {c!r}\n")

💡 **Key Takeaway** — Semantic chunking groups sentences by meaning instead of a fixed size: boundaries fall exactly where the topic shifts, verified by comparing sentence embeddings. The cost is real — one embedding call per document during chunking, instead of a free string-slicing operation. Note: the rest of today's pipeline (Sections 3–7) keeps using `chunk_text`/`ALL_CHUNKS` from before — semantic chunking is shown here as an alternative technique, not swapped into the pipeline the rest of the lab builds on.

---

## Section 3 — Embeddings & Vector Indexing

**Goal:** Understand what an embedding represents and build a searchable index from the chunks.

An embedding is a fixed-length numeric vector positioned so that semantically similar text ends up close together in vector space. That's what makes semantic search possible — we're not matching keywords, we're matching meaning.

### Guided Demo

In [ ]:
CHUNK_SIZE = 300
OVERLAP = 50

ALL_CHUNKS = []
for doc_id, doc in UNDERWRITING_DOCS.items():
    for i, chunk in enumerate(chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)):
        ALL_CHUNKS.append({"text": chunk, "source": doc_id, "title": doc["title"], "chunk_index": i})

print(f"Total chunks across all documents: {len(ALL_CHUNKS)}")

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("underwriting_docs")

chunk_texts = [c["text"] for c in ALL_CHUNKS]
chunk_embeddings = client.embed(chunk_texts)

collection.add(
    ids=[f"{c['source']}-{c['chunk_index']}" for c in ALL_CHUNKS],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[{"source": c["source"], "title": c["title"], "chunk_index": c["chunk_index"]} for c in ALL_CHUNKS],
)

print("Collection count:", collection.count())
print("Embedding vector length:", len(chunk_embeddings[0]))
print("Example embedding:", chunk_embeddings[0][:60], "...")

&nbsp;

### 🔧 Try It Yourself — Exercise 3.1

Fetch one item back out of the collection by ID (e.g. `"GL-GUIDELINES-0"`) using `collection.get(ids=[...], include=["documents", "metadatas", "embeddings"])` and print its stored document text, metadata, and the length of its embedding. Confirm it matches what you'd expect from `ALL_CHUNKS`.

In [ ]:
record = collection.get(ids=["GL-GUIDELINES-0"], include=["documents", "metadatas", "embeddings"])
print("Document:", record["documents"][0])
print("Metadata:", record["metadatas"][0])
print("Embedding length:", len(record["embeddings"][0]))

&nbsp;

💡 **Key Takeaway** — An embedding is a fixed-length numeric vector. You can't read it, but the *distance* between two embeddings tells you how semantically similar the underlying text is — that's the entire mechanism behind semantic search.

---

## Section 4 — Retrieval

**Goal:** Retrieve the most relevant chunks for a real question and see how K affects what comes back.

To retrieve, we embed the query with the *same* embedding model used to embed the chunks, then ask the vector store for the K nearest chunks.

### Guided Demo

In [ ]:
def retrieve(query: str, k: int = 3):
    query_embedding = client.embed([query])[0]
    return collection.query(query_embeddings=[query_embedding], n_results=k)

In [ ]:
query = "What construction types are preferred for property coverage?"

for k in [1, 3, 5]:
    results = retrieve(query, k=k)
    print(f"--- k={k} ---")
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"[{meta['source']} #{meta['chunk_index']}] distance={dist:.3f}  {doc[:100]}...")
    print()

**What is that "distance" number?** Chroma ranks results by computing a *distance* between the query embedding and each stored chunk embedding — a single number that says how far apart two vectors are in embedding space. Lower distance means the two pieces of text are more semantically similar; higher distance means less similar.

A couple of things worth knowing before you see these numbers:
- Chroma's default distance metric is **squared L2 (Euclidean) distance** — straight-line distance between the two vectors, squared — unless a collection is explicitly created with `metadata={"hnsw:space": "cosine"}` to use cosine distance instead. Our `underwriting_docs` collection uses the default (L2).
- The raw number isn't a percentage or a "confidence score" — it has no fixed scale you can eyeball in isolation. What matters is the *ranking*: the chunk with the smallest distance for a given query is the model's best match, and distances are only meaningful relative to each other within the same query.

You'll see these distance values printed directly in the retrieval results below, and again in Section 6 when a query has no good match in the documents at all.

&nbsp;

### 🔧 Try It Yourself — Exercise 4.1

Try at least 2 of your own queries against the index. For each, compare `k=1` vs `k=5` — does the extra context at `k=5` help, or does it start pulling in chunks from unrelated documents?

#### Query 1

In [ ]:
my_query_1 = "What are the cyber coverage exclusions related to unpatched systems?"

for k in [1, 5]:
    results = retrieve(my_query_1, k=k)
    print(f"--- k={k} ---")
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"[{meta['source']} #{meta['chunk_index']}] distance={dist:.3f}  {doc}")
    print()

#### Query 2 — your turn

In [ ]:
my_query_2 = "YOUR QUERY HERE"

for k in [1, 5]:
    results = retrieve(my_query_2, k=k)
    print(f"--- k={k} ---")
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"[{meta['source']} #{meta['chunk_index']}] distance={dist:.3f}  {doc[:100]}...")
    print()

&nbsp;

💡 **Key Takeaway** — Retrieval quality depends on both the embedding model and K. Too small a K misses relevant context; too large dilutes the prompt with irrelevant chunks.

---

## Section 5 — Build the Complete RAG Pipeline

**Goal:** Chain every prior step into one working pipeline and see retrieval's actual effect on answer quality.

The online pipeline: query → embed → retrieve top-K → build an augmented prompt (retrieved chunks + question) → `generate(...)`.

*Where would reranking fit?* A production system often retrieves a larger candidate set (e.g. K=20) with a fast vector search, then runs a slower, more accurate reranking model over just those candidates to reorder them before picking the final top-K to put in the prompt. We're not implementing that today — with only 4 documents there's nothing to rerank — but it's the next lever to pull once retrieval quality plateaus at scale.

### Guided Demo

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are an underwriting assistant answering questions about company guidelines."
    "You will recieve a user's question inside <USER_QUESTION></USER_QUESTION> tags."
    "You'll also receive fetched context from internal documents to help answer the question, provided in <DOC_CONTEXT></DOC_CONTEXT> wrappers."
    "These documents are provided by the system, not the user. Do not refer to them as the user's documents"
    "If the provided doc context does not contain the answer, explictly say so and do not answer the question."
    "Politely decline any non-insurance related questions."
)
def generate_augmented_answer(query: str, k: int = 3) -> str:
    results = retrieve(query, k=k)
    context = "\n\n".join(
        f"[Source: {meta['source']} — {meta['title']}]\n{doc}"
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    )

    augmented_prompt = (
        "Use the following underwriting guideline excerpts to answer the question. "
        "If the excerpts don't contain the answer, say so explicitly — do not guess.\n\n"
        f"<DOC_CONTEXT>\n{context}</DOC_CONTEXT>\n\n"
        f"<USER_QUESTION> {query}</USER_QUESTION>"
    )
    return client.generate(augmented_prompt, system_prompt=RAG_SYSTEM_PROMPT).text, results

question = "Can we offer property coverage above $2,000,000 in a flood zone without documentation?" 

#### No retrieval

In [ ]:
no_retrieval = client.generate(question, system_prompt=RAG_SYSTEM_PROMPT)
print(no_retrieval.text)

#### With retrieval

In [ ]:
answer, context = generate_augmented_answer(question)
print(answer)

In [ ]:
print("Retrieved context: ", context)

&nbsp;

### 🔧 Try It Yourself — Exercise 5.1

Run `generate_augmented_answer` on 2 more questions about the underwriting documents (e.g. about workers' comp classification or cyber exclusions). For each, also run the same question through plain `client.generate(...)` with no retrieved context. Compare the two answers side by side — is the retrieval-grounded answer more specific and accurate?

#### Question 1

In [ ]:
question_1 = "What's the max out of state travel duration for worker's comp?"

**No retrieval:**

In [ ]:
print(client.generate(question_1, system_prompt=RAG_SYSTEM_PROMPT).text)

**With retrieval:**

In [ ]:
answer, context = generate_augmented_answer(question_1)
print(answer)

In [ ]:
print(context)

#### Question 2

In [ ]:
question_2 = "I'm underwriting a general liability policy for the first time. What do I need to know?"

**No retrieval:**

In [ ]:
print(client.generate(question_2, system_prompt=RAG_SYSTEM_PROMPT).text)

**With retrieval:**

In [ ]:
answer, context = generate_augmented_answer(question_2)
print(answer)

In [ ]:
print(context)

&nbsp;

### 🔧 Try It Yourself — Exercise 5.2 — Prompt Engineering for Citations

Look back at the "With retrieval" answers you just printed for Question 1 and Question 2 — the model answers correctly, but gives you no way to tell *which document* it pulled the answer from. In a real underwriting workflow, an unsupported answer isn't very useful — the underwriter needs to verify it against the source.

Every retrieved chunk is now tagged with its document title and code (`[Source: DOC-ID — Title]`) before it's handed to the model — but the model has no instruction to actually use it. Edit `RAG_SYSTEM_PROMPT` below to instruct the model to cite its source(s) in the final answer — e.g. by including the doc code in brackets (`[GL-GUIDELINES]`) or the document title. Then re-run the cell below it to see if it worked.

*Hint: something like "Cite the source document code(s) in brackets, e.g. [GL-GUIDELINES], immediately after any claim drawn from the context."*

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are an underwriting assistant answering questions about company guidelines."
    "You will recieve a user's question inside <USER_QUESTION></USER_QUESTION> tags."
    "You'll also receive fetched context from internal documents to help answer the question, provided in <DOC_CONTEXT></DOC_CONTEXT> wrappers."
    "These documents are provided by the system, not the user. Do not refer to them as the user's documents"
    "If the provided doc context does not contain the answer, explictly say so and do not answer the question."
    "Politely decline any non-insurance related questions."
    "[CITATION INSTRUCTIONS HERE]"
)

In [ ]:
answer, results = generate_augmented_answer(question_2)
print(answer)

&nbsp;

💡 **Key Takeaway** — Retrieved context measurably grounds the answer: the LLM answers specifically instead of guessing, hedging, or refusing.

---

## Section 6 — Failure Modes & Tuning

**Goal:** Recognize that RAG isn't automatically correct — it fails in specific, recognizable ways. Debugging a bad answer means checking each pipeline stage independently: chunking, retrieval, or generation.

### Failure mode 1 — irrelevant retrieval

None of our documents mention parking policy. A vector search still returns its "closest" matches — they just won't be relevant.

In [ ]:
off_topic_query = "What is the company's parking policy for employees?"
results = retrieve(off_topic_query, k=3)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"[{meta['source']} #{meta['chunk_index']}] distance={dist:.3f}")
    print(f"  {doc}")

In [ ]:
answer, context = generate_augmented_answer(off_topic_query)
print(answer)

&nbsp;

### Failure mode 2 — chunk size too small

Compare this to Section 2 — an aggressively small chunk size can split a rule across chunks so no single chunk contains the complete idea.

In [ ]:
tiny_chunks = chunk_text(UNDERWRITING_DOCS["WORKCOMP-CLASSIFICATION"]["text"], chunk_size=40, overlap=0)
print(f"{len(tiny_chunks)} tiny chunks\n")
for i, c in enumerate(tiny_chunks[:6]):
    print(f"[{i}] {c!r}")

&nbsp;

### Failure mode 3 — hallucination despite relevant context

The cyber exclusions document never states a specific dollar deductible. Watch whether the model says so — or invents a plausible-sounding number anyway.

In [ ]:
question = "What is the exact dollar deductible for cyber coverage?"
print(generate_augmented_answer(question)[0])

💡 **Key Takeaway** — Bad answers can originate from chunking, retrieval, or generation — and the fix is different for each. Debugging RAG means checking each stage independently rather than only tweaking the final prompt.

---

## Section 7 — Wrap-Up

Today you built both halves of a RAG pipeline:

**Offline:** documents → chunk → embed → store in Chroma
**Online:** query → embed → retrieve top-K → augment the prompt → generate

You saw that chunk size, K, and the retrieved context itself all shape answer quality — and that even with good retrieval, the model can still hallucinate on questions the documents don't fully answer.

💬 **Reflection** — Looking back across Sections 2–6, which single change (chunk size, K, or prompt wording) had the biggest effect on answer quality for you?

**Next session:** the retrieval pipeline built today becomes an input to tool-calling and agent workflows — Session 3 combines retrieved context with tools so an agent can decide *when* to retrieve, not just answer a single fixed query.